# V1 Classical CNN Experiments

This notebook defines the three November-2018-compatible experiments used for the genuine Kalluri dataset run. All variants use the same leakage-safe train/validation partition, seed 42, 150 × 150 input, Adam, categorical cross-entropy, batch size 32, and 20 epochs. The original source test split is not opened here.

The downloaded dataset already contains source-provided offline transformations. Here, *augmentation* means additional online `ImageDataGenerator` augmentation during training. Outputs are intentionally cleared so no third-party fruit photographs are embedded in Git.

In [ ]:
from __future__ import print_function

import json
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras.callbacks import ModelCheckpoint
from keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from keras.models import Sequential
from keras.preprocessing.image import ImageDataGenerator


In [ ]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 20
DATASET_ROOT = os.path.join('..', 'dataset')
TRAIN_ROOT = os.path.join(DATASET_ROOT, 'train')
VALIDATION_ROOT = os.path.join(DATASET_ROOT, 'validation')
RESULTS_ROOT = os.path.join('..', 'results')
CHECKPOINT_ROOT = os.path.join('..', 'model', 'experiments')

for directory in (RESULTS_ROOT, CHECKPOINT_ROOT):
    if not os.path.isdir(directory):
        os.makedirs(directory)

def reset_seeds():
    random.seed(SEED)
    np.random.seed(SEED)
    tf.set_random_seed(SEED)


## Shared model and data helpers

Experiment 1 omits dropout. Experiments 2 and 3 insert `Dropout(0.5)` before the six-class softmax layer. Only Experiment 3 adds online augmentation.

In [ ]:
def build_model(dropout_rate):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    if dropout_rate:
        model.add(Dropout(dropout_rate))
    model.add(Dense(6, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def make_generators(use_online_augmentation):
    if use_online_augmentation:
        training_data = ImageDataGenerator(
            rescale=1.0 / 255,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest',
        )
    else:
        training_data = ImageDataGenerator(rescale=1.0 / 255)
    validation_data = ImageDataGenerator(rescale=1.0 / 255)
    training_generator = training_data.flow_from_directory(
        TRAIN_ROOT, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=True, seed=SEED,
    )
    validation_generator = validation_data.flow_from_directory(
        VALIDATION_ROOT, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False,
    )
    return training_generator, validation_generator


In [ ]:
def save_curves(history, experiment_number):
    for metric, title, ylabel in (
        ('acc', 'Accuracy', 'Accuracy'),
        ('loss', 'Loss', 'Loss'),
    ):
        plt.figure(figsize=(8, 5))
        plt.plot(history.history[metric], label='Training {}'.format(title))
        plt.plot(history.history['val_' + metric], label='Validation {}'.format(title))
        plt.title('Experiment {:02d}: Training and Validation {}'.format(experiment_number, title))
        plt.xlabel('Epoch')
        plt.ylabel(ylabel)
        plt.legend()
        plt.tight_layout()
        path = os.path.join(RESULTS_ROOT, 'experiment_{:02d}_{}.png'.format(experiment_number, metric.replace('acc', 'accuracy')))
        plt.savefig(path)
        plt.close()

def run_experiment(experiment_number, name, dropout_rate, use_online_augmentation):
    reset_seeds()
    training_generator, validation_generator = make_generators(use_online_augmentation)
    model = build_model(dropout_rate)
    model.summary()
    checkpoint_path = os.path.join(CHECKPOINT_ROOT, 'experiment_{:02d}_best.h5'.format(experiment_number))
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_acc', mode='max', save_best_only=True, verbose=1)
    history = model.fit_generator(
        training_generator,
        steps_per_epoch=int(math.ceil(training_generator.samples / float(BATCH_SIZE))),
        epochs=EPOCHS,
        validation_data=validation_generator,
        validation_steps=int(math.ceil(validation_generator.samples / float(BATCH_SIZE))),
        callbacks=[checkpoint],
        verbose=2,
    )
    save_curves(history, experiment_number)
    best_index = int(np.argmax(history.history['val_acc']))
    summary = {
        'experiment': experiment_number,
        'name': name,
        'dropout': dropout_rate,
        'online_augmentation': use_online_augmentation,
        'best_epoch': best_index + 1,
        'best_train_accuracy': float(history.history['acc'][best_index]),
        'best_validation_accuracy': float(history.history['val_acc'][best_index]),
        'best_validation_loss': float(history.history['val_loss'][best_index]),
        'best_accuracy_gap': float(history.history['acc'][best_index] - history.history['val_acc'][best_index]),
        'history': dict((key, [float(value) for value in values]) for key, values in history.history.items()),
    }
    with open(os.path.join(RESULTS_ROOT, 'experiment_{:02d}_summary.json'.format(experiment_number)), 'w') as handle:
        json.dump(summary, handle, indent=2, sort_keys=True)
        handle.write('\n')
    return summary


## Execute exactly three variants

These cells intentionally state each experimental change explicitly. Run them once, in order, after preparing the documented local partition.

In [ ]:
experiment_1 = run_experiment(
    1, 'Custom CNN baseline', dropout_rate=0.0, use_online_augmentation=False
)

In [ ]:
experiment_2 = run_experiment(
    2, 'Custom CNN with dropout', dropout_rate=0.5, use_online_augmentation=False
)

In [ ]:
experiment_3 = run_experiment(
    3, 'Custom CNN with dropout and augmentation',
    dropout_rate=0.5, use_online_augmentation=True,
)

## Validation-only selection

The genuine run selected Experiment 2 at epoch 8: 90.00% validation accuracy and 0.5937 validation loss. Experiment 3 reached 88.89%, and the baseline reached 87.41%. No test metric was consulted for this decision. The one-time original-test evaluation is recorded in `EXPERIMENTS.md`; it is deliberately not repeated by this notebook.

In [ ]:
validation_ranking = sorted(
    (experiment_1, experiment_2, experiment_3),
    key=lambda item: (
        -item['best_validation_accuracy'],
        item['best_validation_loss'],
        abs(item['best_accuracy_gap']),
        item['experiment'],
    ),
)
print('Selected from validation only: Experiment {}'.format(validation_ranking[0]['experiment']))

Perfect numerical reproducibility is not guaranteed across every CPU/GPU implementation, even with Python, NumPy, and TensorFlow 1.x seeds fixed.